# Wan 2.2 TI2V-5B Video Generation on AWS Trainium2 (trn2.3xlarge)

End-to-end text-to-video and image-to-video generation with **Wan 2.2 TI2V-5B** (5B dense) on a single `trn2.3xlarge` instance.

## Environment

| Component | Value |
|-----------|-------|
| **DLAMI** | Deep Learning AMI Neuron (Ubuntu 24.04) 20260410 |
| **Neuron SDK** | 2.29 |
| **Instance** | trn2.3xlarge (LNC=2, 4 logical cores, 96 GB HBM total) |
| **venv** | `/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/` |

## Architecture

- **Model**: [Wan 2.2 TI2V-5B](https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B-Diffusers) -- 5B dense transformer, supports both T2V and I2V
- **Resolution**: 640x480 (480P), 81 frames, 50 denoising steps
- **Parallelism**: TP=4 across all 4 cores (world_size=4)
- **Key optimizations**:
  - NKI Flash Attention kernel
  - `--enable-ccop-compute-overlap` for ~17% transformer speedup
  - Stateful rolling VAE decoder (on-device HBM cache, zero host transfer)
  - `local_rms_norm` workaround for compiler issue with DistributedRMSNorm

## Pipeline Flow

```
1. Load pipeline on CPU, encode text with UMT5-XXL
2. Compile & load text encoder (TP=4) -> encode prompt on Neuron
3. Compile & load transformer (TP=4) -> 50 denoising steps
4. Compile & load VAE decoder (rolling cache, bfloat16) -> decode latents
5. Export video
```

## Compilation Code

This notebook uses compilation and inference code from [Henan's aws-neuron-samples fork](https://github.com/whn09/aws-neuron-samples/tree/master/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v).

**Expected total time**: ~50 min compilation + ~60s inference (480P)

## Step 1: Environment Setup & Verification

In [1]:
import os
import subprocess
import sys

# ============================================================
# Configuration -- adjust these for your setup
# ============================================================

# Storage -- trn2.3xlarge has 1x 470 GB NVMe SSD
NVME_MOUNT = "/opt/dlami/nvme"
WORK_DIR = NVME_MOUNT
MODELS_DIR = f"{WORK_DIR}/models"
MODEL_NAME = "Wan2.2-TI2V-5B-Diffusers"
MODEL_DIR = f"{MODELS_DIR}/{MODEL_NAME}"
COMPILED_DIR = f"{WORK_DIR}/compiled_ti2v_5b_tp4"
COMPILER_WORKDIR = f"{WORK_DIR}/compiler_workdir"
CACHE_DIR = f"{WORK_DIR}/wan2.2_ti2v_hf_cache_dir"  # Must match hardcoded path in compile scripts
SAMPLES_DIR = f"{WORK_DIR}/aws-neuron-samples"
HENAN_DIR = f"{SAMPLES_DIR}/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
VENV = "/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference"

# Parallelism -- trn2.3xlarge with LNC=2 has 4 logical cores
TP_DEGREE = 4
WORLD_SIZE = 4  # TP only, no CP on 4 cores

# Video generation
HEIGHT = 480
WIDTH = 640
NUM_FRAMES = 81
NUM_STEPS = 50
FPS = 16

print(f"TP={TP_DEGREE}, World Size={WORLD_SIZE}")
print(f"Resolution: {WIDTH}x{HEIGHT}, {NUM_FRAMES} frames, {NUM_STEPS} steps")
print(f"Work directory: {WORK_DIR}")

TP=4, World Size=4
Resolution: 640x480, 81 frames, 50 steps
Work directory: /opt/dlami/nvme


In [2]:
# Verify Neuron devices -- expect 1 device with 4 cores at LNC=2
!neuron-ls

instance-type: trn2.3xlarge
instance-id: i-004b3f47e995863b1
logical-neuroncore-config: 2
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 4      | 0-3      | 96 GB  | 0000:33:00.0 | 0-11     | 0    |
+--------+--------+----------+--------+--------------+----------+------+


In [3]:
# Verify SDK versions -- expect SDK 2.29
!pip show neuronx-cc torch-neuronx neuronx-distributed 2>/dev/null | grep -E 'Name:|Version:'

Name: neuronx-cc
Version: 2.24.5133.0+58f8de22


Name: torch-neuronx
Version: 2.9.0.2.13.24727+8e870898


Name: neuronx-distributed
Version: 0.18.27753+1cafd54f


In [4]:
# Verify disk space (need ~80 GB for model + compiled artifacts)
!df -h / | head -2

Filesystem      Size  Used Avail Use% Mounted on
/dev/root       290G   68G  223G  24% /


In [5]:
%%bash -e
# Mount NVMe storage (trn2.3xlarge has 1x 470 GB NVMe SSD)
NVME_MOUNT="/opt/dlami/nvme"

if mountpoint -q "${NVME_MOUNT}" 2>/dev/null; then
    echo "NVMe already mounted at ${NVME_MOUNT}"
else
    # Find NVMe devices (exclude root EBS)
    NVME_DEVICES=$(lsblk -d -n -o NAME,TYPE | grep disk | grep nvme | awk '{print "/dev/"$1}' | sort)
    ROOT_DEV=$(findmnt -n -o SOURCE / | sed 's/p[0-9]*$//' | sed 's/[0-9]*$//')

    NVME_TO_MOUNT=""
    for dev in $NVME_DEVICES; do
        if [[ "$dev" != "$ROOT_DEV"* ]] && ! lsblk -n -o MOUNTPOINT "$dev" | grep -q '/'; then
            NVME_TO_MOUNT="$dev"
            break
        fi
    done

    if [ -z "$NVME_TO_MOUNT" ]; then
        echo "No unmounted NVMe found, creating directory on EBS instead"
        sudo mkdir -p "${NVME_MOUNT}"
        sudo chown ubuntu:ubuntu "${NVME_MOUNT}"
    else
        echo "Formatting and mounting ${NVME_TO_MOUNT}..."
        sudo mkfs.ext4 -F "$NVME_TO_MOUNT"
        sudo mkdir -p "${NVME_MOUNT}"
        sudo mount "$NVME_TO_MOUNT" "${NVME_MOUNT}"
        sudo chown ubuntu:ubuntu "${NVME_MOUNT}"
        echo "Mounted at ${NVME_MOUNT}"
    fi
fi

df -h ${NVME_MOUNT}

NVMe already mounted at /opt/dlami/nvme


Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme1n1    430G  142G  267G  35% /opt/dlami/nv

me


In [6]:
# Create working directories
for d in [MODELS_DIR, COMPILED_DIR, COMPILER_WORKDIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)
print("Directories created.")

Directories created.


## Step 2: Install Dependencies

In [7]:
# Install required packages
!pip install -q diffusers accelerate imageio-ffmpeg ftfy

In [8]:
# Patch diffusers: nearest-exact -> nearest (required for Trainium2)
import diffusers

vae_path = os.path.join(
    os.path.dirname(diffusers.__file__),
    "models", "autoencoders", "autoencoder_kl_wan.py"
)
with open(vae_path) as f:
    content = f.read()
if "nearest-exact" in content:
    content = content.replace("nearest-exact", "nearest")
    with open(vae_path, "w") as f:
        f.write(content)
    print("Patched diffusers: nearest-exact -> nearest")
else:
    print("Diffusers already patched")

Diffusers already patched


## Step 3: Clone Neuron Samples & Patch for trn2.3xlarge

Henan's code targets trn2.48xlarge (8 NeuronCores, world_size=8). We need to patch the
run script to work with trn2.3xlarge (4 cores, world_size=4).

The compile scripts already support world_size=4 via command-line arguments, but the
inference script `run_wan2.2_ti2v.py` has hardcoded `NEURON_RT_NUM_CORES=8` and
`LOCAL_WORLD_SIZE=8` that must be changed.

In [9]:
%%bash -e
SAMPLES_DIR="/opt/dlami/nvme/aws-neuron-samples"
HENAN_DIR="${SAMPLES_DIR}/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"

if [ -d "${HENAN_DIR}" ]; then
    echo "aws-neuron-samples already cloned"
else
    echo "Cloning aws-neuron-samples (Henan's fork)..."
    git clone --depth 1 https://github.com/whn09/aws-neuron-samples.git "${SAMPLES_DIR}"
fi

echo "TI2V-5B compilation code:"
ls ${HENAN_DIR}/neuron_wan2_2_ti2v/

aws-neuron-samples already cloned
TI2V-5B compilation code:


__pycache__
cache_hf_model.py
compile_decoder.py
compile_decoder_nocache.py
compile_decoder_rolling.

py
compile_encoder.py
compile_text_encoder.py
compile_transformer.py
distributed_rmsnorm.py
neuron_c

ommons.py
neuron_parallel_utils.py


In [10]:
# Patch run_wan2.2_ti2v.py for trn2.3xlarge (4 cores instead of 8)
#
# The run script hardcodes NEURON_RT_NUM_CORES=8 and LOCAL_WORLD_SIZE=8.
# On trn2.3xlarge with LNC=2 we only have 4 logical cores.
# The compiled models read their world_size from config.json, so this
# patch makes the runtime match our 4-core compilation.

run_script = os.path.join(HENAN_DIR, "run_wan2.2_ti2v.py")

with open(run_script) as f:
    content = f.read()

patched = False

# Patch NEURON_RT_NUM_CORES from 8 to 4
if '"NEURON_RT_NUM_CORES"] = "8"' in content:
    content = content.replace(
        '"NEURON_RT_NUM_CORES"] = "8"',
        '"NEURON_RT_NUM_CORES"] = "4"'
    )
    patched = True

# Patch LOCAL_WORLD_SIZE from 8 to 4
if '"LOCAL_WORLD_SIZE"] = "8"' in content:
    content = content.replace(
        '"LOCAL_WORLD_SIZE"] = "8"',
        '"LOCAL_WORLD_SIZE"] = "4"'
    )
    patched = True

if patched:
    with open(run_script, "w") as f:
        f.write(content)
    print("Patched run_wan2.2_ti2v.py: NEURON_RT_NUM_CORES=4, LOCAL_WORLD_SIZE=4")
else:
    print("run_wan2.2_ti2v.py already patched or has different format")

# Verify the patch
!grep -n 'NEURON_RT_NUM_CORES\|LOCAL_WORLD_SIZE' {run_script}

run_wan2.2_ti2v.py already patched or has different format
17:    NEURON_RT_NUM_CORES=8 python run_wan2.2_ti2v.py --compiled_models_dir compiled_models
21:os.environ["NEURON_RT_NUM_CORES"] = "4"
22:os.environ["LOCAL_WORLD_SIZE"] = "4"


## Step 4: Download Model (~34 GB)

Downloads the full Wan 2.2 TI2V-5B model from HuggingFace. This includes the UMT5-XXL text encoder (~11 GB), the 5B transformer (~20 GB), and the VAE (~500 MB).

In [11]:
if os.path.exists(os.path.join(MODEL_DIR, "model_index.json")):
    print(f"Model already downloaded at {MODEL_DIR}")
else:
    print(f"Downloading Wan-AI/Wan2.2-TI2V-5B-Diffusers (~34 GB)...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        "Wan-AI/Wan2.2-TI2V-5B-Diffusers",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
    )
    print("Download complete.")

!du -sh {MODEL_DIR}

Model already downloaded at /opt/dlami/nvme/models/Wan2.2-TI2V-5B-Diffusers
32G	/opt/dlami/nvme/models/Wan2.2-TI2V-5B-Diffusers


In [12]:
# Cache model for Henan's compilation scripts.
# Henan's cache_hf_model.py hardcodes CACHE_DIR to /opt/dlami/nvme/wan2.2_ti2v_hf_cache_dir
# We do it inline to be explicit about the path.

import torch
from diffusers import AutoencoderKLWan, WanPipeline

if not os.path.exists(CACHE_DIR) or len(os.listdir(CACHE_DIR)) == 0:
    print(f"Caching model to {CACHE_DIR}...")
    vae = AutoencoderKLWan.from_pretrained(
        "Wan-AI/Wan2.2-TI2V-5B-Diffusers",
        subfolder="vae", torch_dtype=torch.float32, cache_dir=CACHE_DIR)
    pipe = WanPipeline.from_pretrained(
        "Wan-AI/Wan2.2-TI2V-5B-Diffusers",
        vae=vae, torch_dtype=torch.bfloat16, cache_dir=CACHE_DIR)
    del vae, pipe
    print("Cache complete.")
else:
    print(f"Model already cached at {CACHE_DIR}")

Model already cached at /opt/dlami/nvme/wan2.2_ti2v_hf_cache_dir


## Step 5: Compile Models

We compile 4 artifacts for Neuron:
1. **Text encoder** (UMT5-XXL, TP=4) -- ~2-3 min
2. **Transformer** (5B, TP=4, world_size=4) -- ~5-8 min
3. **VAE decoder** (rolling cache, bfloat16, world_size=4) -- ~40 min
4. **post_quant_conv** (float32, world_size=4) -- ~1 min

### Key Optimization: `--enable-ccop-compute-overlap`

This flag enables compute/communication overlap in the compiler, providing ~17% transformer speedup.

**Important**: This flag must be passed directly via `compiler_args`, NOT via the `NEURON_CC_FLAGS` environment variable (which is silently ignored by ModelBuilder).

### Adapting for trn2.3xlarge (4 cores)

Henan's original code targets trn2.48xlarge with TP=4, CP=2 (world_size=8). On trn2.3xlarge with 4 cores, we use:
- **TP=4, world_size=4** (no CP; CP degree is inferred as `world_size / tp_degree = 1`)
- The text encoder also uses `--world_size 4` (must match transformer)
- **All models must share the same world_size=4**, including the VAE decoder and post_quant_conv. NxDModel segfaults if models with different world_sizes are loaded in the same process.

In [13]:
# Set up Python path for compilation modules
os.environ["NEURON_RT_VIRTUAL_CORE_SIZE"] = "2"
os.environ["PYTHONPATH"] = HENAN_DIR + ":" + os.environ.get("PYTHONPATH", "")

# Required for NKI kernels (not optional -- NKI fails without this)
os.environ["XLA_DISABLE_FUNCTIONALIZATION"] = "1"

# These env vars were set in ALL benchmark runs (baseline and optimized).
# They were never independently A/B tested on TI2V-5B, but removing them
# would make results non-comparable to the measured benchmarks.
os.environ["NEURON_FUSE_SOFTMAX"] = "1"
os.environ["NEURON_CUSTOM_SILU"] = "1"

print(f"PYTHONPATH includes: {HENAN_DIR}")
print("Env vars set: XLA_DISABLE_FUNCTIONALIZATION, NEURON_FUSE_SOFTMAX, NEURON_CUSTOM_SILU")

PYTHONPATH includes: /opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v
Env vars set: XLA_DISABLE_FUNCTIONALIZATION, NEURON_FUSE_SOFTMAX, NEURON_CUSTOM_SILU


### 5a: Compile Text Encoder (UMT5-XXL, TP=4)

In [14]:
%%bash -e
HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
COMPILED_DIR="/opt/dlami/nvme/compiled_ti2v_5b_tp4"
export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2

if [ -f "${COMPILED_DIR}/text_encoder/nxd_model.pt" ]; then
    echo "Text encoder already compiled"
else
    echo "Compiling text encoder (TP=4, world_size=4)..."
    # IMPORTANT: Do NOT use neuron_parallel_compile -- it overrides world_size to 8
    python \
        ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_text_encoder.py \
        --max_sequence_length 512 \
        --tp_degree 4 --world_size 4 \
        --compiled_models_dir ${COMPILED_DIR} \
        2>&1 | tail -15
    echo "Text encoder compiled."
fi

ls -lh ${COMPILED_DIR}/text_encoder/nxd_model.pt

Text encoder already compiled


-rw-rw-r-- 1 ubuntu ubuntu 199M Apr 24 04:59 /opt/dlami/nvme/compiled_ti2v_5b_tp4/text_encoder/nxd_m

odel.pt


### 5b: Compile Transformer (5B, TP=4, world_size=4)

This is the main compilation step. The transformer has 30 DiT layers with NKI flash attention.

**Note on trn2.3xlarge**: With only 4 cores, we set `--world_size 4 --tp_degree 4`. The
compile script infers CP degree as `world_size / tp_degree = 1`, meaning no context
parallelism. All CP communication ops (scatter, all-gather) are cleanly skipped when
`context_parallel_enabled = False`.

CFG parallelism also requires world_size=8, so it is not available on 4 cores.

In [15]:
%%bash -e
HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
COMPILED_DIR="/opt/dlami/nvme/compiled_ti2v_5b_tp4"
COMPILER_WORKDIR="/opt/dlami/nvme/compiler_workdir"
export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2
export NEURON_FUSE_SOFTMAX=1
export NEURON_CUSTOM_SILU=1
export XLA_DISABLE_FUNCTIONALIZATION=1

if [ -f "${COMPILED_DIR}/transformer/nxd_model.pt" ]; then
    echo "Transformer already compiled"
else
    echo "Compiling transformer (TP=4, world_size=4, inferred CP=1)..."
    echo "This will take ~5-8 minutes..."
    # IMPORTANT: Do NOT use neuron_parallel_compile -- it overrides world_size to 8
    python \
        ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_transformer.py \
        --height 480 --width 640 \
        --num_frames 81 --max_sequence_length 512 \
        --tp_degree 4 --world_size 4 \
        --compiled_models_dir ${COMPILED_DIR} \
        --compiler_workdir ${COMPILER_WORKDIR}/transformer \
        2>&1 | tail -15
    echo "Transformer compiled."
fi

ls -lh ${COMPILED_DIR}/transformer/nxd_model.pt

Transformer already compiled


-rw-rw-r-- 1 ubuntu ubuntu 32M Apr 24 05:01 /opt/dlami/nvme/compiled_ti2v_5b_tp4/transformer/nxd_mod

el.pt


### 5c: Compile VAE Decoder (Rolling Cache, bfloat16) + post_quant_conv

The rolling cache decoder maintains 34 temporal cache tensors on-device (HBM) between calls,
enabling flicker-free video without host<->device transfer overhead.

Uses `--model-type=unet-inference` since the VAE is Conv3D-heavy (not attention-based).

**Critical: world_size must match other models.** Even though the decoder doesn't need TP, NxDModel
segfaults if a world_size=1 model is loaded after world_size=4 models. The decoder and PQC must
be compiled with `--tp_degree 4 --world_size 4` (weights are duplicated across all cores).

At 640x480, the decoder NEFF exceeds the default 5M instruction limit (5.67M instructions).
We pass `--max_instruction_limit 6000000` to raise the limit. At 720P, even this is not enough
and tiled spatial decode is required instead.

In [16]:
%%bash -e
HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
COMPILED_DIR="/opt/dlami/nvme/compiled_ti2v_5b_tp4"
COMPILER_WORKDIR="/opt/dlami/nvme/compiler_workdir"
export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2
export NEURON_FUSE_SOFTMAX=1
export NEURON_CUSTOM_SILU=1
export XLA_DISABLE_FUNCTIONALIZATION=1

# Compile rolling-cache decoder + post_quant_conv together
# CRITICAL: --world_size 4 --tp_degree 4 must match transformer/text_encoder
# NxDModel segfaults if world_size=1 models are loaded alongside world_size=4 models
if [ -f "${COMPILED_DIR}/decoder_rolling/nxd_model.pt" ]; then
    echo "Rolling cache decoder already compiled"
else
    echo "Compiling VAE decoder (rolling cache, bfloat16) + post_quant_conv..."
    echo "This will take ~40 minutes at 640x480..."
    # IMPORTANT: Do NOT use neuron_parallel_compile -- it overrides world_size to 8
    python \
        ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_decoder_rolling.py \
        --height 480 --width 640 \
        --tp_degree 4 --world_size 4 \
        --compiled_models_dir ${COMPILED_DIR} \
        --compiler_workdir ${COMPILER_WORKDIR}/decoder \
        --cache_dir /opt/dlami/nvme/wan2.2_ti2v_hf_cache_dir \
        --max_instruction_limit 6000000 \
        2>&1 | tail -15
    echo "Decoder + post_quant_conv compiled."
fi

echo "Decoder:"
ls -lh ${COMPILED_DIR}/decoder_rolling/nxd_model.pt
echo "post_quant_conv:"
ls -lh ${COMPILED_DIR}/post_quant_conv/nxd_model.pt

Rolling cache decoder already compiled
Decoder:


-rw-rw-r-- 1 ubuntu ubuntu 239M Apr 24 06:48 /opt/dlami/nvme/compiled_ti2v_5b_tp4/decoder_rolling/nx

d_model.pt


post_quant_conv:


-rw-rw-r-- 1 ubuntu ubuntu 9.4M Apr 24 06:48 /opt/dlami/nvme/compiled_ti2v_5b_tp4/post_quant_conv/nx

d_model.pt


### 5d: Verify All Compiled Artifacts

In [17]:
import json

# Artifact paths -- these match what Henan's scripts produce
artifacts = {
    "text_encoder": f"{COMPILED_DIR}/text_encoder/nxd_model.pt",
    "transformer": f"{COMPILED_DIR}/transformer/nxd_model.pt",
    "decoder_rolling": f"{COMPILED_DIR}/decoder_rolling/nxd_model.pt",
    "post_quant_conv": f"{COMPILED_DIR}/post_quant_conv/nxd_model.pt",
}

print("Compiled artifacts:")
all_ok = True
for name, path in artifacts.items():
    exists = os.path.exists(path)
    if exists:
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  {name:20s}: {size_mb:>8.1f} MB")
    else:
        print(f"  {name:20s}: MISSING")
        all_ok = False

# Show transformer config
config_path = f"{COMPILED_DIR}/transformer/config.json"
if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    print(f"\nTransformer config: TP={config.get('tp_degree')}, "
          f"CP={config.get('cp_degree', 1)}, "
          f"WS={config.get('world_size')}")

assert all_ok, "Some artifacts are missing! Re-run compilation steps."
print("\nAll artifacts present.")

Compiled artifacts:
  text_encoder        :    198.4 MB
  transformer         :     31.4 MB
  decoder_rolling     :    238.2 MB
  post_quant_conv     :      9.3 MB

Transformer config: TP=4, CP=1, WS=4

All artifacts present.


## Step 6: Run Inference

We use Henan's `run_wan2.2_ti2v.py` script (patched in Step 3 for 4 cores) which
orchestrates the full pipeline:
1. Loads compiled text encoder, transformer, and decoder onto Neuron cores
2. Encodes the prompt with UMT5-XXL
3. Runs 50 denoising steps with CFG (classifier-free guidance)
4. Decodes latents to video frames using the rolling-cache VAE

The run script reads `world_size`, `tp_degree`, and `cp_degree` from each compiled
model's `config.json`, so it automatically picks up our 4-core configuration.

### Text-to-Video (T2V)

In [18]:
%%bash -e
HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
COMPILED_DIR="/opt/dlami/nvme/compiled_ti2v_5b_tp4"

export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2
export NEURON_RT_NUM_CORES=4
export LOCAL_WORLD_SIZE=4
export NEURON_FUSE_SOFTMAX=1
export NEURON_CUSTOM_SILU=1
export XLA_DISABLE_FUNCTIONALIZATION=1

echo "Running Text-to-Video inference..."
echo "Resolution: 640x480, 81 frames, 50 steps"
echo ""

python ${HENAN_DIR}/run_wan2.2_ti2v.py \
    --compiled_models_dir ${COMPILED_DIR} \
    --height 480 --width 640 \
    --num_frames 81 \
    --num_inference_steps 50 \
    --prompt "A cat walks through a sunlit garden, realistic style, 4K quality" \
    --negative_prompt "blurred, low quality, static, overexposed" \
    --output /opt/dlami/nvme/output_t2v_480p.mp4 \
    --fps 16 \
    2>&1

Running Text-to-Video inference...


Resolution: 640x480, 81 frames, 50 steps



/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/mod

ules/moe/blockwise.py:100: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit inst

ead.
  component, error = import_nki(config)


...

 98%|█████████▊| 49/50 [00:51<00:01,  1.04s/it]

100%|██████████| 50/50 [00:52<00:00,  1.04s/it]

100%|██████████| 50/50 [00:52<00:00,  1.04s/it]


Main inference: 59.96s (1.199s/step)

Output frames: 84

T2V inference time: 59.96s
Per step (denois

e only): 1.199s

Video saved to: /opt/dlami/nvme/output_t2v_480p.mp4


### Display Generated Video

In [19]:
import io
from pathlib import Path
from IPython.display import Video, Image as IPImage, display
from PIL import Image

video_path = f"{WORK_DIR}/output_t2v_480p.mp4"

if Path(video_path).exists():
    file_size = Path(video_path).stat().st_size / (1024 * 1024)
    print(f"Video: {video_path} ({file_size:.1f} MB)")
    
    # Display video (works in Jupyter Lab)
    display(Video(video_path, embed=True, width=640))
else:
    print(f"Video not found at {video_path}")

Video: /opt/dlami/nvme/output_t2v_480p.mp4 (1.7 MB)


### Image-to-Video (I2V) -- Optional

The TI2V-5B model also supports image-to-video generation. Provide an input image and the model will animate it.

**Note**: The VAE encoder for I2V runs on CPU (Neuron compiler issue NCC_IBIR158). This adds a small overhead but only runs once per video.

In [20]:
# Uncomment and run this cell for Image-to-Video generation
# First, place your input image at /opt/dlami/nvme/input.png

# %%bash -e
# HENAN_DIR="/opt/dlami/nvme/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_ti2v"
# COMPILED_DIR="/opt/dlami/nvme/compiled_ti2v_5b_tp4"
# 
# export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
# export NEURON_RT_VIRTUAL_CORE_SIZE=2
# export NEURON_RT_NUM_CORES=4
# export LOCAL_WORLD_SIZE=4
# export NEURON_FUSE_SOFTMAX=1
# export NEURON_CUSTOM_SILU=1
# export XLA_DISABLE_FUNCTIONALIZATION=1
# 
# python ${HENAN_DIR}/run_wan2.2_ti2v.py \
#     --compiled_models_dir ${COMPILED_DIR} \
#     --image /opt/dlami/nvme/input.png \
#     --height 480 --width 640 \
#     --num_frames 81 \
#     --num_inference_steps 50 \
#     --prompt "A cat walks on the grass, realistic" \
#     --output /opt/dlami/nvme/output_i2v_480p.mp4 \
#     --fps 16

## Step 7: Scaling Up to Higher Resolutions

Once 480P works, you can scale up:

| Resolution | Frames | Notes |
|-----------|--------|-------|
| 512x384 | 81 | Fastest, good for testing |
| **640x480** | **81** | **Default for this notebook** |
| 640x480 | 121 | 5 seconds @ 24fps |
| 1280x704 | 81 | 720P -- requires tiled VAE decode, recompile needed |

To change resolution, you need to **recompile the transformer and decoder** with the new
dimensions. The text encoder does not depend on resolution.

### Recompile for a Different Resolution

```bash
# Example: compile transformer for 512x384
# IMPORTANT: Do NOT use neuron_parallel_compile (overrides world_size)
python \
    ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_transformer.py \
    --height 384 --width 512 \
    --num_frames 81 --max_sequence_length 512 \
    --tp_degree 4 --world_size 4 \
    --compiled_models_dir /opt/dlami/nvme/compiled_512x384

# Example: compile decoder for 512x384
# CRITICAL: --world_size 4 must match transformer (NxDModel segfaults otherwise)
python \
    ${HENAN_DIR}/neuron_wan2_2_ti2v/compile_decoder_rolling.py \
    --height 384 --width 512 \
    --tp_degree 4 --world_size 4 \
    --compiled_models_dir /opt/dlami/nvme/compiled_512x384
```

### 720P (1280x704) -- Tiled VAE Decode

At 720P, the VAE decoder's Conv3D operators exceed the Neuron compiler's per-operator
instruction limit. The solution is to compile the decoder at a smaller tile resolution
(e.g., 384x512) and tile the full-resolution latent at inference time with overlap blending.
Henan's code supports this via `test_resolutions.sh`. On trn2.3xlarge with 4 cores, tiles
are decoded sequentially rather than in parallel, so 720P will be slower than on 48xlarge.

## Performance Notes

### trn2.3xlarge vs trn2.48xlarge

| Metric | trn2.48xlarge (8 cores, WS=8) | trn2.3xlarge (4 cores, WS=4) |
|--------|-------------------------------|------------------------------|
| Cores | 64 (LNC=2) | 4 (LNC=2) |
| Config | TP=4, CP=2, WS=8 | TP=4, WS=4 |
| 480P T2V (50 steps) | ~18-33s | **~60s** (measured) |
| CFG Parallel | Yes (WS=8) | No (need WS=8) |
| Tiled VAE (720P) | 8 parallel tiles | Sequential tiles |

### Optimization Summary

Tested on TI2V-5B with before/after measurements:

| Optimization | Before | After | Delta | Tested? |
|-------------|--------|-------|-------|----------|
| `--enable-ccop-compute-overlap` | 254.0 ms | 211.2 ms | **-16.9%** | A/B tested |
| `--cc-pipeline-tiling-factor=4` | 211.2 ms | 208.8 ms | -1.1% | A/B tested |
| `-O2` vs `-O1` | 208.8 ms | 208.6 ms | -0.1% | A/B tested |
| Rolling cache VAE | N/A | N/A | Eliminates ~960 MB host transfer | Structural |
| NKI Flash Attention | N/A | N/A | Required for compilation | Structural |
| `local_rms_norm` | N/A | N/A | Required (compiler workaround) | Structural |

**Part of all benchmark runs but never independently toggled**:
- `NEURON_FUSE_SOFTMAX=1` -- set in every run; removing would invalidate comparisons
- `NEURON_CUSTOM_SILU=1` -- same; note: known to cause NCC_IBIR182 on some other models' VAE decoders

`XLA_DISABLE_FUNCTIONALIZATION=1` is not an optimization -- it is functionally required for NKI kernels.

### What Did NOT Help (A/B tested)

| Approach | Result | Why |
|----------|--------|-----|
| NKI fused o_proj kernel | +6.5% regression | Serializes all-reduce, prevents ccop overlap |
| ModuleMarkerWrapper | +23.1% regression | Breaks compiler's recursive layer detection |
| QKV fusion | +5.0% regression | Compiler already optimizes sequential matmuls |
| `-O3` | No improvement | Tested across multiple Neuron projects |

## Troubleshooting

### Segfault or hang when loading decoder after transformer
**All NxDModels loaded in the same process must share the same world_size.** If the transformer
was compiled with world_size=4 but the decoder with world_size=1, NxDModel.to_neuron() will
segfault or hang when loading the decoder. Recompile the decoder with `--world_size 4 --tp_degree 4`.

### `neuron_parallel_compile` overrides world_size
`neuron_parallel_compile` sets its own WORLD_SIZE environment variable internally, overriding
any `--world_size` argument. This causes compiled models to have world_size=8 even when
you pass `--world_size 4`. **Always use `python` directly, never `neuron_parallel_compile`.**

### `run_wan2.2_ti2v.py` fails with core allocation error
The script has hardcoded `NEURON_RT_NUM_CORES=8`. The patching cell in Step 3 changes this
to 4. If you re-cloned the repo, re-run the patching cell.

### "nearest-exact" interpolation error
Run the diffusers patch cell (Step 2) -- this replaces `nearest-exact` with `nearest` in the VAE.

### Replica groups assertion error
This is the `DistributedRMSNorm` compiler issue. Henan's code uses `local_rms_norm` as a workaround.

### Out of memory during compilation
trn2.3xlarge has 128 GB system RAM. If compilation OOMs, try:
- Reducing swap usage
- Using `WORLD_SIZE=4` (not 8) for all compilations

### Slow first import
On fresh DLAMI instances, the first import of torch-neuronx can take up to 5 minutes
(library rehydration). This is expected.

### Decoder exceeds instruction limit (NCC_EVRF007)
At 640x480, the decoder generates 5.67M instructions, exceeding the default 5M limit.
Pass `--max_instruction_limit 6000000` to `compile_decoder_rolling.py`.

In [21]:
print("Notebook complete.")
print(f"\nEnvironment summary:")
print(f"  DLAMI: Deep Learning AMI Neuron (Ubuntu 24.04) 20260410")
print(f"  SDK: 2.29")
print(f"  Instance: trn2.3xlarge (LNC=2, 4 cores)")
print(f"  Model: Wan 2.2 TI2V-5B (5B dense)")
print(f"  Resolution: {WIDTH}x{HEIGHT}, {NUM_FRAMES}f, {NUM_STEPS} steps")
print(f"  Parallelism: TP={TP_DEGREE}, WS={WORLD_SIZE}")

Notebook complete.

Environment summary:
  DLAMI: Deep Learning AMI Neuron (Ubuntu 24.04) 20260410
  SDK: 2.29
  Instance: trn2.3xlarge (LNC=2, 4 cores)
  Model: Wan 2.2 TI2V-5B (5B dense)
  Resolution: 640x480, 81f, 50 steps
  Parallelism: TP=4, WS=4
